# CRM AI Platform — проверка на GPU (Kaggle)

Блокнот прогоняет **синтетические** примеры через настоящие типы задач AI Platform
(те же промпты, контракты выхода и проверки привязки, что в рабочей платформе) на модели
**Qwen3.5-4B Q4_K_M** под llama.cpp с GPU.

**Правило:** сюда загружаются только придуманные данные. Реальные клиенты, звонки и документы
компаний на Kaggle не отправляются.

### Перед запуском
1. Справа в **Settings**: *Accelerator* → **GPU T4 x2** (или P100), *Internet* → **On**
   (интернет на Kaggle включается после подтверждения телефона в профиле).
2. **Add Input** → ваш приватный датасет с `kaggle-eval-bundle.zip`
   (собирается командой `scripts/build_kaggle_bundle.sh` в репозитории AI Platform).
3. **Run All**. Первый запуск: ~15–20 минут на сборку llama.cpp и загрузку модели (3 ГБ).

Результат — таблица в конце и файл `report.json` во вкладке **Output**.

In [ ]:
# 1. Есть ли GPU
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "GPU не найден: включите Accelerator в Settings")

In [ ]:
# 2. Найти и проверить пакет (evalrun + синтетические наборы)
import glob, hashlib, os, shutil
from pathlib import Path

hits = glob.glob("/kaggle/input/**/MANIFEST.txt", recursive=True)
assert hits, "Пакет не найден: Add Input -> датасет с kaggle-eval-bundle.zip"
src = Path(hits[0]).parent
WORK = Path("/kaggle/working/bundle")
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(src, WORK)
os.chmod(WORK / "evalrun", 0o755)

manifest = (WORK / "MANIFEST.txt").read_text()
print(manifest)
for line in manifest.splitlines():
    parts = line.split()
    if len(parts) == 2 and len(parts[0]) == 64:
        digest = hashlib.sha256((WORK / parts[1]).read_bytes()).hexdigest()
        assert digest == parts[0], f"контрольная сумма не совпала: {parts[1]}"
print("пакет цел:", sorted(p.name for p in (WORK / "datasets").glob("*.jsonl")))

In [ ]:
# 3. Собрать llama.cpp с CUDA — та же версия, что в локальном образе (тег b11960 = 8642541b2)
import os, subprocess
from pathlib import Path

LLAMA = Path("/kaggle/working/llama.cpp")
SERVER = LLAMA / "build/bin/llama-server"
if not SERVER.exists():
    if not LLAMA.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "b11960",
                        "https://github.com/ggml-org/llama.cpp", str(LLAMA)], check=True)
    head = subprocess.run(["git", "-C", str(LLAMA), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    assert head.startswith("8642541b2"), f"неожиданная версия llama.cpp: {head}"
    # 60 = P100, 75 = T4: собираем только под карты Kaggle, так быстрее.
    subprocess.run(["cmake", "-S", str(LLAMA), "-B", str(LLAMA / "build"), "-DGGML_CUDA=ON",
                    "-DCMAKE_CUDA_ARCHITECTURES=60;75", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"], check=True)
    subprocess.run(["cmake", "--build", str(LLAMA / "build"), "--target", "llama-server",
                    "-j", str(os.cpu_count() or 4)], check=True)
print(subprocess.run([str(SERVER), "--version"], capture_output=True, text=True).stderr.strip()[-300:])

In [ ]:
# 4. Скачать модель и проверить sha256 (то же значение, что у локальной копии)
import hashlib
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO, FILE = "bartowski/Qwen_Qwen3.5-4B-GGUF", "Qwen_Qwen3.5-4B-Q4_K_M.gguf"
SHA256 = "13c16f426047e2de38cd075bdade4a7bcbc8c774384876f677740cda65f8a983"
MODEL = Path(hf_hub_download(REPO, FILE, local_dir="/kaggle/working/models"))
h = hashlib.sha256()
with open(MODEL, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 24), b""):
        h.update(chunk)
assert h.hexdigest() == SHA256, "sha256 модели не совпал — файл не тот или повреждён"
print("модель проверена:", MODEL, MODEL.stat().st_size // 2**20, "МБ")

In [ ]:
# 5. Запустить llama-server на GPU (как в compose.llm.yaml, плюс -ngl 99)
import subprocess, time, urllib.request

log = open("/kaggle/working/llama-server.log", "w")
server = subprocess.Popen([str(SERVER), "-m", str(MODEL), "--host", "127.0.0.1", "--port", "8080",
                           "--ctx-size", "8192", "--parallel", "1", "-ngl", "99", "--jinja", "--no-webui"],
                          stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2).status == 200:
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("сервер не поднялся, см. /kaggle/working/llama-server.log")
print("llama-server готов")
print(subprocess.run("grep -iE 'offload|CUDA|VRAM' /kaggle/working/llama-server.log | tail -5",
                     shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 6. Прогнать все синтетические примеры через настоящие типы задач
import subprocess
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.split("\n")[0].strip()
datasets = sorted(str(p) for p in (WORK / "datasets").glob("*.jsonl"))
cmd = [str(WORK / "evalrun"), "-provider", "llamacpp", "-llm", "http://127.0.0.1:8080",
       "-model", f"qwen3.5-4b-q4km@{GPU or 'gpu'}", "-out", "/kaggle/working/report.json", *datasets]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "evalrun завершился с ошибкой"

## Как читать результат
- **ok** — задача прошла контракт и проверки привязки (цитаты есть во входе, числа — из входа).
- **output_not_grounded** — модель написала то, чего нет во входных данных; платформа такой ответ
  **отбросила**. Это и есть «выдуманные числа/факты».
- **checks_failed** — ответ формально верный, но не тот по смыслу: ассистент выбрал не тот
  инструмент (`tool:schedule_meeting`), предложил действия там, где их не должно быть
  (`max_actions`), или не сказал честно «недостаточно данных» (`insufficient_data`).
- **found** — доля ожидаемых слов в ответе: грубая автоматическая подсказка, не оценка качества.
- **median_s / max_s** — время одного ответа на этом GPU.

Смысл ответов (правильно ли объяснено, вежлив ли черновик) нужно прочитать глазами — ниже.

In [ ]:
# 7. Сводная таблица
import json
import pandas as pd

report = json.load(open("/kaggle/working/report.json"))
summary = pd.DataFrame(report["summary"])
summary["errors"] = summary["errors"].apply(lambda e: ", ".join(f"{k}: {v}" for k, v in e.items()) or "—")
summary["by_language"] = summary["by_language"].apply(lambda e: ", ".join(f"{k} {v}" for k, v in sorted(e.items())))
print("модель:", report["model"])
summary[["type", "cases", "succeeded", "first_try_ok", "checks_failed", "contains_found_mean",
         "latency_median_s", "latency_max_s", "errors", "by_language"]]

In [ ]:
# 8. Ответы модели — прочитать глазами
import textwrap

def show(r):
    head = f"■ {r['id']} [{r['type']}, {r['language']}] {r['status']} {r.get('error_code', '')} {r['latency_s']} с"
    print(head)
    if r.get("missed"):
        print("  не найдено:", r["missed"])
    out = r.get("output")
    if out:
        out = {k: v for k, v in out.items() if k != "generator"}
        print(textwrap.indent(json.dumps(out, ensure_ascii=False, indent=1)[:2500], "  "))
    elif r.get("error"):
        print("  ", r["error"][:500])
    print()

for r in report["results"]:
    show(r)

## 9. (Необязательно) Whisper: распознавание речи

Загружайте **только свою собственную тестовую запись** (например, прочитайте вслух короткий
придуманный диалог на русском и на кыргызском). Записи реальных звонков сюда не загружаются.

Добавьте аудио (`.wav`, `.mp3`, `.m4a`, `.ogg`) отдельным приватным датасетом через **Add Input**
и запустите ячейку ниже. Она покажет распознанный текст, определённый язык и время.
Если Whisper плохо справится с кыргызским — это тоже важный результат.

In [ ]:
import glob, subprocess, sys, time
audio = [p for ext in ("wav", "mp3", "m4a", "ogg") for p in glob.glob(f"/kaggle/input/**/*.{ext}", recursive=True)]
if not audio:
    print("Аудио не добавлено — пропускаем.")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper==1.1.1"], check=True)
    from faster_whisper import WhisperModel
    w = WhisperModel("large-v3", device="cuda", compute_type="float16")
    for path in audio:
        t = time.time()
        segments, info = w.transcribe(path, beam_size=5)
        text = " ".join(s.text.strip() for s in segments)
        print(f"■ {path}\n  язык: {info.language} ({info.language_probability:.2f}), длительность {info.duration:.0f} с, "
              f"распознано за {time.time() - t:.1f} с\n  {text}\n")

In [ ]:
# 10. Остановить сервер. report.json и llama-server.log — во вкладке Output.
server.terminate()
print("готово: скачайте /kaggle/working/report.json")